# Cross-Model Comparison & Evaluation of the 7 Baselines
**Project:** Group Activity Recognition — Volleyball Baselines (Ibrahim et al., CVPR 2016)

### Evaluation Scope
This notebook aggregates and compares the evaluation metrics from all **7 requested models** on the official Volleyball test split:
1. **Model 1 (B1):** Image Classification (Single Frame ResNet-50)
2. **Model 2 (B3):** Fine-tuned Person Classification (Max-pooled Person Crops)
3. **Model 3 (B4):** Temporal Model with Image Features (Donahue et al. LRCN)
4. **Model 4 (B5):** Temporal Model with Person Features (ImageNet Features + LSTM)
5. **Model 5 (B6):** Two-stage Model without LSTM 1 (Fine-tuned Person Features + Group LSTM 2)
6. **Model 6 (B7):** Two-stage Model without LSTM 2 (Person LSTM 1 Temporal Features + Max-pool)
7. **Model 7:** Two-stage Hierarchical Deep Temporal Model (Full Proposed Model)

> **Explicit Confirmation:** Baseline 2 (B2 — Person Classification) is intentionally excluded from this comparison per project requirements.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

RESULTS_BASE = "results"
COMP_DIR = os.path.join(RESULTS_BASE, "cross_model_comparison")
PLOTS_DIR = os.path.join(COMP_DIR, "plots")
METRICS_DIR = os.path.join(COMP_DIR, "metrics")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

MODELS_INFO = [
    ("1_B1_Image_Classification", "Model 1: B1 - Image Classification"),
    ("2_B3_Finetuned_Person_Classification", "Model 2: B3 - Fine-tuned Person Classification"),
    ("3_B4_Temporal_Model_Image_Features", "Model 3: B4 - Temporal Model with Image Features"),
    ("4_B5_Temporal_Model_Person_Features", "Model 4: B5 - Temporal Model with Person Features"),
    ("5_B6_Two_Stage_No_LSTM1", "Model 5: B6 - Two-stage Model without LSTM 1"),
    ("6_B7_Two_Stage_No_LSTM2", "Model 6: B7 - Two-stage Model without LSTM 2"),
    ("7_Model7_Hierarchical_Full", "Model 7: Two-stage Hierarchical Model (Full)")
]

In [ ]:
# Aggregating Metrics from Individual Notebook Runs
records = []

for folder_name, display_name in MODELS_INFO:
    metrics_path = os.path.join(RESULTS_BASE, folder_name, "metrics", "metrics.json")
    if os.path.isfile(metrics_path):
        with open(metrics_path, 'r') as f:
            data = json.load(f)
            data["Model"] = display_name
            data["Folder"] = folder_name
            records.append(data)
    else:
        records.append({
            "Model": display_name,
            "Folder": folder_name,
            "test_accuracy": np.nan,
            "test_f1_weighted": np.nan,
            "test_f1_macro": np.nan,
            "test_precision": np.nan,
            "test_recall": np.nan,
            "best_val_accuracy": np.nan,
            "status": "Awaiting Execution on GPU/Kaggle"
        })

df_comparison = pd.DataFrame(records)
print("=== Consolidated Model Comparison Table ===")
print(df_comparison[['Model', 'test_accuracy', 'test_f1_weighted', 'test_f1_macro', 'best_val_accuracy']])

# Export consolidated table to CSV & JSON
df_comparison.to_csv(os.path.join(METRICS_DIR, "cross_model_metrics.csv"), index=False)
df_comparison.to_json(os.path.join(METRICS_DIR, "cross_model_metrics.json"), orient="records", indent=4)
print(f"\nConsolidated metrics saved to {METRICS_DIR}")

In [ ]:
# Cross-Model Comparison Visualizations
valid_df = df_comparison.dropna(subset=['test_accuracy'])

if not valid_df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(valid_df['Model'], valid_df['test_accuracy'] * 100, color='royalblue', edgecolor='navy')
    ax.set_xlabel('Test Accuracy (%)')
    ax.set_title('Cross-Model Comparison: Group Activity Recognition Accuracy', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, f"{width:.1f}%", va='center')
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "accuracy_comparison.png"), dpi=300)
    plt.show()
else:
    print("Note: Run the individual notebooks first to populate test metrics across all models.")